In [ ]:
!pip install chromadb sentence-transformers pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.8/20.8 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.6/326.6 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.3/132.3 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.0/208.0 kB 13.5 MB/s eta 

In [ ]:
import os, re, uuid    # uuid : 고유 식별 id 생성용
from typing import List   # type hint 기능 제공 (가독성)
from sentence_transformers import SentenceTransformer   # 문장 단위의 의미 임베딩 라이브러리
from chromadb import PersistentClient
import pypdf

PDF_PATH = "sample.pdf"
CHROMA_DIR = ".chroma_pdf_demo"
COLLECTION = "pdf_docs"
MODEL_NAME = "all-MiniLM-L6-v2"


In [ ]:
def read_pdfFunc(path:str) -> str:
  if not os.path.exists(path):
    raise FileNotFoundError(f"허걱 pdf 파일이 없어요:{path}")

  text_pages = []  # 페이지별 텍스트 저장 리스트
  try:
    with open(path, 'rb') as f:
      reader = pypdf.PdfReader(f)
      for i, page in enumerate(reader.pages):
        txt = page.extract_text() or ""
        text_pages.append(txt)
    return "\n\n".join(text_pages)
  except Exception:
    raise RuntimeError(f"pdf 추출 실패")

# 문단 단위로 분리
def split_paragraphFunc(text:str, min_len:int=40) -> List[str]:
  chunks = re.split(r"\n\s*\n+", text)   # 빈 줄 기준 문단 분리
  chunks = [re.sub(r"\s+", " ", p).strip() for p in chunks]
  return [p for p in chunks if len(p) >= min_len]

def embedderFunc(name:str=MODEL_NAME):
  return SentenceTransformer(name)

def embedFunc(model, texts:List[str])-> List[List[float]]:
  return model.encode(texts, normalize_embeddings=True).tolist()

def get_collectionFunc(chroma_dir:str, name:str):
  client = PersistentClient(path=chroma_dir)
  return client.get_or_create_collection(name)

# pdf 파일을 읽어 VectorDb에 저장
def upsert_pdfFunc(pdf_path:str):
  full_text = read_pdfFunc(pdf_path)
  # print(full_text)

  if not full_text.strip():
    print('pdf에서 추출된 자료가 없음')
    return 0

  chunks = split_paragraphFunc(full_text, min_len=40)
  # print(len(chunks))   # 2
  if not chunks:
    print('저장할 문단이 없어요')
    return 0

  model = embedderFunc(MODEL_NAME)
  embs = embedFunc(model, chunks)
  # print(embs)

  metas = []
  for c in chunks:
    metas.append(
        {
            "source":os.path.basename(pdf_path),
            "len":len(c)
        }
    )

  collection = get_collectionFunc(CHROMA_DIR, COLLECTION)
  ids = [str(uuid.uuid4()) for _ in chunks]
  collection.add(ids=ids, documents=chunks, embeddings=embs, metadatas=metas)
  return len(chunks)

def searchFunc(query:str, k:int):
  model = embedderFunc(MODEL_NAME)
  q_emb = embedFunc(model, [query])
  collection = get_collectionFunc(CHROMA_DIR, COLLECTION)
  res = collection.query(query_embeddings=q_emb, n_results=k)

  docs = res.get('documents', [[]])[0]    # 예외 방지용 패턴
  metas = res.get('metadatas', [[]])[0]
  ids = res.get('ids', [[]])[0]
  dists = res.get('distances', [[]])[0]

  for i, (doc, meta, _id, dist) in enumerate(zip(docs, metas, ids, dists)):
    print(f'\n[{i}] id={_id}')
    print(f'source={meta.get('source')}, len={meta.get('len')}, distance={dist:.4f}')
    print(doc[:300] + ("..." if len(doc) > 300 else ""))

In [ ]:
if __name__ == "__main__":
  # n = upsert_pdfFunc(PDF_PATH)

  # print(f"\n저장된 문단 수 : {n}")
  searchFunc("강남에 지역 문화적 뉘앙스를 더하면 로컬 개발자들의 개성이 살아날 수 있다",  k=3)


[0] id=2307782b-74b0-465c-974b-79a98410ab46
source=sample.pdf, len=332, distance=0.7237
복도에는 전세계 곳곳의 풍경을 담은 사진이 걸려 있다. 영어타운 수업은 한 학기에 한번 교실을 벗어나 지역으로 향한다. 마을 장터에서 오란다와 고추장을 만들고 박물관이나 아이스링크를 체험하며 영어를 몸으로 익힌다. 원어민 교사와 함께하는 문화체험에 신난 아이들은 “영어가 진짜 재미있어요!”라고 말한다. 박 원장은 “영천영어타운은 지역 내 유일한 원어민 영어교육 공간”이라며 “면 단위 학교 학생들도 이용할 수 있도록 스쿨버스 2 대를 운행해 접근성을 높이고 있다”고 말했다. 그리고 이렇게 덧붙였다. “폐교가 다시 살아난 이곳에서 아이...

[1] id=3eb1ace6-d634-430d-b392-83f932cda0f1
source=sample.pdf, len=697, distance=0.7423
데이터와 직관 결합한 전략 글로벌 전략과 각 나라 니즈의 균형을 맞추는 방법에 대해 슈라이어 사장은 "데이터와 과학, 그리고 감성과 직관의 조합"이라고 답했다. 시청자 그룹, 시청 패턴, 시장별 취향 데이터를 면밀히 보지만, 동시에 각 지역 임원들과 크리에이터들의 본능적 감각을 신뢰한다는 것이다. "역사를 보면 진정한 성공작은 대부분 예상 밖의 시도에서 나왔다. '스타워즈', ' 더 베어'도 그랬다. 샌드위치 가게 얘기가 이렇게 흥행할 줄 누가 알았겠느냐." 그는 크리에이터들이 모험할 수 있는 환경을 만드는 것이 중요하다고 역설했다...

[2] id=090d981c-87eb-4436-99ed-cc1acaabe97b
source=sample.pdf, len=317, distance=0.7573
8 명이 일한다. 원장은 박광일 영천교육지원청 교육지원과장이 겸직하고 있다. 이곳은 영천시 내 18 개 초등학교 학생의 영어교육을 담당한다. 매일 오전엔 4∼6 학년을 대상으로 정규 수업을 진행하고, 오후엔 3∼6 학년까지 